# Kafka Demo — Lab 3

### Connect to Kafka Broker Server
Open an SSH tunnel in your terminal and leave it running while you use this notebook.
Replace `<NetID>` with your UIC NetID:
```
ssh -o ServerAliveInterval=60 -L 9092:localhost:9092 <NetID>@cs544-f26.cs.uic.edu -NTf
```

### To kill connection
```
lsof -ti:9092 | xargs kill -9
```

### Setup
```
python -m pip install kafka-python
```

See [bug_list.md](./bug_list.md) for frequent bugs and solutions.

In [9]:
import os
from datetime import datetime
from json import dumps, loads
from time import sleep
from random import randint
from kafka import KafkaConsumer, KafkaProducer

# Update this for your own recitation section :)
topic = 'recitation-tstra6' # replace x with your recitation section

### Producer Mode -> Writes Data to Broker

In [10]:
# Create a producer to write data to kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html

# [TODO]: Replace '...' with the address of your Kafka bootstrap server
producer = KafkaProducer(bootstrap_servers=["localhost:9092"],
                        value_serializer=lambda x: dumps(x).encode('utf-8'))

# [TODO]: Add cities of your choice
cities = ["chicago", "seattle", "portland"]

# Write data via the producer
print("Writing to Kafka Broker")
for i in range(10):
    data = f'{datetime.now().strftime("%Y-%m-%d %H:%M:%S")},{cities[randint(0,len(cities)-1)]},{randint(18, 32)}ºC'
    print(f"Writing: {data}")
    producer.send(topic=topic, value=data)
    sleep(1)

Writing to Kafka Broker
Writing: 2026-09-25 16:15:47,seattle,18ºC
Writing: 2026-09-25 16:15:48,chicago,24ºC
Writing: 2026-09-25 16:15:49,seattle,26ºC
Writing: 2026-09-25 16:15:50,portland,26ºC
Writing: 2026-09-25 16:15:51,chicago,26ºC
Writing: 2026-09-25 16:15:52,portland,26ºC
Writing: 2026-09-25 16:15:53,portland,28ºC
Writing: 2026-09-25 16:15:54,chicago,29ºC
Writing: 2026-09-25 16:15:55,seattle,20ºC
Writing: 2026-09-25 16:15:56,seattle,29ºC


### Consumer Mode -> Reads Data from Broker

In [7]:
# Create a consumer to read data from kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html

# [TODO]: Complete the missing ... parameters/arguments using the Kafka documentation
consumer = KafkaConsumer(
    topic,
    bootstrap_servers=["localhost:9092"],
    auto_offset_reset="earliest", #Experiment with different values
    # Commit that an offset has been read
    enable_auto_commit=True,
    # How often to tell Kafka, an offset has been read
    auto_commit_interval_ms=1000
)

print('Reading Kafka Broker')
for message in consumer:
    message = message.value.decode()
    # Default message.value type is bytes!
    print(loads(message))
    os.system(f"echo {message} >> kafka_log.csv")

Reading Kafka Broker
2026-09-23 21:29:53,seattle,19ºC
2026-09-23 21:29:54,chicago,29ºC
2026-09-23 21:29:55,chicago,30ºC
2026-09-23 21:29:56,portland,28ºC
2026-09-23 21:29:57,portland,23ºC
2026-09-23 21:29:58,seattle,25ºC
2026-09-23 21:29:59,seattle,19ºC
2026-09-23 21:30:00,seattle,31ºC
2026-09-23 21:30:01,portland,28ºC
2026-09-23 21:30:02,portland,27ºC


KeyboardInterrupt: 

# Use kcat!
It's a CLI (Command Line Interface). Previously known as kafkacat


Ref: https://docs.confluent.io/platform/current/app-development/kafkacat-usage.html

In [ ]:
#kcat command: connect to local Kafka broker, specify a topic, and consume messages from the earliest offset
kcat -b localhost:9092 -t recitation-tstra6 -C -o beginning